# 에이전트 하네스 — 보안 취약점 월간 보고서

앞 노트북에서는 **도구를 가진 에이전트 하나**를 만들었습니다.
이번에는 에이전트가 **여러 단계를 스스로 계획하고, 파일을 주고받고, 일을 나눠 맡기고,
회사 양식을 읽어 PPT 보고서까지** 만듭니다. 이렇게 갖춰진 실행 환경을 **하네스(harness)** 라고 합니다.

## 하는 일

```
input/cve_list.md  →  조사  →  위험도 평가  →  output/report.pptx
                       (서브에이전트들이 분업)   (회사 양식대로)
```

## 이번에 배우는 네 가지

| 개념 | 한 줄 설명 | 단계 |
|---|---|---|
| **파일시스템(샌드박스)** | 정해진 폴더 안에서만 읽고 쓴다 | Step 2 |
| **planning** | 할 일 목록을 **스스로 세우고 지워 나간다** | Step 3 |
| **서브에이전트** | 전문 역할에게 **일을 떼어 맡긴다** | Step 4 |
| **스킬(SKILL.md)** | 회사 양식을 **파일로 가르친다** | Step 5 |

## 왜 필요한가

도구만으로는 한계가 있습니다.

| 문제 | 하네스의 답 |
|---|---|
| 단계가 많아 중간에 길을 잃는다 | planning — 할 일을 적어 두고 확인 |
| 중간 결과를 주고받을 곳이 없다 | 파일시스템 — 파일로 넘긴다 |
| 한 에이전트가 모든 걸 잘할 수 없다 | 서브에이전트 — 역할을 나눈다 |
| 지시문이 길어져 감당이 안 된다 | 스킬 — 필요할 때만 읽어 간다 |

> 실행 순서: 위에서부터 차례대로. `agent/` 폴더에서 실행하세요.

## Step 0. 준비

```bash
uv add deepagents python-pptx
```

`deepagents`는 LangChain 위에 **하네스**를 얹어 주는 패키지입니다.
`create_deep_agent` 하나가 네 기능을 모두 담고 있습니다.

In [1]:
import shutil
from pathlib import Path

from dotenv import load_dotenv
from langchain.tools import tool
from langchain.agents.middleware import TodoListMiddleware
from deepagents import create_deep_agent, SubAgent
from deepagents.backends import FilesystemBackend

load_dotenv(override=True)

MODEL = "openai:gpt-5.4-mini"
WORK = Path("layer4_work")          # 에이전트가 쓸 수 있는 유일한 폴더
print("모델:", MODEL)
print("작업 폴더:", WORK.resolve())

모델: openai:gpt-5.4-mini
작업 폴더: D:\ai_edu_main\llm_edu\agent\layer4_work


### 응답을 읽는 작은 도우미

`create_deep_agent`의 답변은 **문자열일 수도, 블록 목록일 수도** 있습니다.
매번 분기하면 지저분하니 함수로 빼 둡니다.

In [2]:
def answer_text(out) -> str:
    """마지막 메시지의 글자만 꺼낸다."""
    c = out["messages"][-1].content
    if isinstance(c, str):
        return c
    return "".join(b.get("text", "") for b in c if isinstance(b, dict))


def tool_names(out) -> list[str]:
    """이번 실행에서 부른 도구 이름을 순서대로."""
    return [c["name"] for m in out["messages"]
            if getattr(m, "tool_calls", None) for c in m.tool_calls]

## Step 1. 작업 폴더와 입력 자료 만들기

하네스에게 줄 **작업 공간**을 준비합니다. 이 폴더가 곧 샌드박스의 경계입니다.

In [3]:
if WORK.exists():
    shutil.rmtree(WORK)            # 매번 깨끗한 상태에서 시작
(WORK / "input").mkdir(parents=True)
(WORK / "output").mkdir()

(WORK / "input" / "cve_list.md").write_text("""# 2026년 9월 점검 대상
| CVE | 제품 | 사내 사용 버전 |
|---|---|---|
| CVE-2021-44228 | Apache Log4j | 2.14.1 |
| CVE-2024-20337 | Cisco AnyConnect | 4.9.0 |
| CVE-2023-4863 | libwebp (Chrome) | 116.0 |
""", encoding="utf-8")

print("준비된 파일:")
for p in sorted(WORK.rglob("*")):
    if p.is_file():
        print("  ", p.relative_to(WORK).as_posix())

준비된 파일:
   input/cve_list.md


## Step 2. 파일시스템 — 샌드박스

```python
create_deep_agent(backend=FilesystemBackend(root_dir="layer4_work"))
```

이 한 줄로 에이전트는 **파일을 다루는 도구들을 자동으로 얻습니다.**
우리가 `read_file`이나 `write_file`을 만들 필요가 없습니다.

In [4]:
backend = FilesystemBackend(root_dir=str(WORK))

basic_agent = create_deep_agent(
    model=MODEL,
    backend=backend,
    system_prompt="너는 보안 분석가다. 한국어로 간결히 답하라.",
)

out = basic_agent.invoke({"messages": [{"role": "user", "content":
    "작업 폴더에 어떤 파일이 있는지 보고, input/cve_list.md 의 CVE 번호만 나열해줘."}]})

print("부른 도구:", tool_names(out))
print()
print(answer_text(out)[:300])

부른 도구: ['ls', 'ls', 'read_file']

작업 폴더 파일:
- `/input/cve_list.md`

`input/cve_list.md`의 CVE 번호:
- CVE-2021-44228
- CVE-2024-20337
- CVE-2023-4863


`ls`로 둘러보고 `read_file`로 읽었습니다. **우리가 만들지 않은 도구들**입니다.

하네스가 기본으로 주는 도구는 아홉 개입니다.

| 도구 | 하는 일 |
|---|---|
| `ls` `glob` `grep` | 찾기 |
| `read_file` `write_file` `edit_file` `delete` | 읽고 쓰기 |
| `execute` | 명령 실행 |
| `task` | **서브에이전트에게 일 맡기기** (Step 4) |

### 샌드박스 경계 확인

`root_dir` 밖은 어떻게 될까요? **직접 시험해 보는 것**이 가장 확실합니다.

In [5]:
out = basic_agent.invoke({"messages": [{"role": "user", "content":
    r"C:\Windows\win.ini 파일을 읽어서 첫 줄을 알려줘."}]})
print(answer_text(out)[:250])

직접 읽을 수 있는 경로 형식이 아니라서 실패했습니다.  
이 환경에서는 `C:\Windows\win.ini` 같은 Windows 절대경로를 지원하지 않고, `/...` 형태의 가상 경로만 가능합니다.

원하시면 파일이 있는 실제 가상 경로를 알려주시면 첫 줄을 읽어드리겠습니다.


읽지 못합니다. 에이전트에게 보이는 세상은 `layer4_work/` 가 전부입니다.

**이것이 샌드박스의 핵심입니다.** 에이전트는 실수하거나, 프롬프트 인젝션에 속거나,
의도치 않은 파일을 지울 수 있습니다. 할 수 있는 범위를 **미리 좁혀 두면**
무엇을 잘못해도 피해가 그 폴더 안에 머뭅니다.

> 15장의 "최소 권한" 원칙이 코드 한 줄로 구현된 것입니다.

## Step 3. Planning — 스스로 할 일을 적는다

단계가 많아지면 에이전트는 길을 잃습니다. 세 번째 CVE를 조사하다가
처음에 뭘 하려 했는지 잊어버리는 식입니다.

`TodoListMiddleware`를 붙이면 에이전트가 **할 일 목록을 만들고, 끝낼 때마다 상태를 바꿉니다.**
사람이 포스트잇을 붙였다 떼는 것과 같습니다.

In [6]:
planning_agent = create_deep_agent(
    model=MODEL,
    backend=backend,
    middleware=[TodoListMiddleware()],      # <- 이 한 줄
    system_prompt="너는 보안 분석가다. 여러 단계 작업은 먼저 할 일 목록을 만들고 진행하라.",
)

out = planning_agent.invoke({"messages": [{"role": "user", "content":
    "input/cve_list.md 의 CVE 3건을 조사하고 위험도를 매겨 output/risk.md 로 정리해줘."}]})

print("남은 상태 키:", [k for k in out if k != "messages"])
print()
for t in out.get("todos", []):
    mark = "[x]" if t["status"] == "completed" else "[ ]"
    print(f"  {mark} {t['content']}")

남은 상태 키: ['todos']

  [x] CVE 목록 파일과 관련 맥락 확인
  [x] 각 CVE의 취약점/영향/패치 여부 조사
  [x] 위험도 산정 및 근거 정리
  [x] output/risk.md 파일 생성


할 일이 **상태와 함께** 남아 있습니다. 두 가지 쓸모가 있습니다.

1. **에이전트 자신**이 매 턴 이 목록을 보고 다음에 뭘 할지 정합니다
2. **우리**가 어디까지 진행됐는지 확인할 수 있습니다 — 긴 작업에서 특히 중요합니다

> planning은 "더 똑똑하게" 만드는 기능이 아니라 **길을 잃지 않게** 하는 기능입니다.
> 단계가 두세 개뿐이라면 없어도 됩니다.

In [7]:
print((WORK / "output" / "risk.md").read_text(encoding="utf-8")[:500])

# CVE 위험도 조사 결과

대상: `input/cve_list.md`

## 위험도 요약

| CVE | 제품 | 사내 사용 버전 | 위험도 | 판단 근거 | 조치 권고 |
|---|---|---:|---|---|---|
| CVE-2021-44228 | Apache Log4j | 2.14.1 | 매우 높음 | Log4Shell로 널리 알려진 JNDI 기반 원격 코드 실행 취약점. Log4j 2.14.1은 영향 대상 버전으로 알려져 있음. 공개 악용 사례가 매우 많음. | 즉시 패치/업그레이드 |
| CVE-2024-20337 | Cisco AnyConnect | 4.9.0 | 높음 | Cisco AnyConnect/ Secure Client 계열의 보안 취약점으로, 구버전 사용 시 영향 가능성이 높음. 다만 세부 영향 범위는 제품 구성과 Cisco 권고문 대조가 필요함. | 우선 업데이트 및 Cisco 권고 확인 |
| CVE-2023-4863 | libwebp (Chrome)


## Step 4. 서브에이전트 — 일을 떼어 맡기기

CVE 조사와 위험도 평가는 **다른 일**입니다. 조사는 사실을 모으는 일이고,
평가는 기준에 비춰 판단하는 일입니다. 한 에이전트에게 둘 다 시키면
지시문이 길어지고 서로 간섭합니다.

서브에이전트는 **각자 다른 지시문을 가진 작은 에이전트**입니다.
부모는 `task` 도구로 일을 넘기고 결과만 받습니다.

In [8]:
researcher = SubAgent(
    name="researcher",
    description="CVE 하나를 조사해 개요·영향·조치를 정리한다. 조사할 CVE 번호를 넘겨라.",
    system_prompt="너는 CVE 조사원이다. 사실만 모아 간결한 markdown 으로 답하라. 위험도는 매기지 마라.",
)

analyst = SubAgent(
    name="analyst",
    description="조사 결과를 받아 위험도(높음/보통/낮음)를 매긴다.",
    system_prompt="너는 위험도 평가자다. 근거 한 줄과 등급만 답하라. 새로운 사실을 지어내지 마라.",
)

print("서브에이전트 2명 정의 완료")

서브에이전트 2명 정의 완료


### 왜 `description`이 중요한가

부모 에이전트는 **`description`만 보고** 누구에게 맡길지 정합니다.
서브에이전트의 지시문(`system_prompt`)은 보지 못합니다.

도구의 독스트링과 같은 역할입니다 — **부모에게 주는 설명서**입니다.

In [9]:
team_agent = create_deep_agent(
    model=MODEL,
    backend=backend,
    subagents=[researcher, analyst],        # <- 이 한 줄
    system_prompt=("너는 보안팀 리더다. 조사는 researcher 에게, 위험도 평가는 analyst 에게 맡겨라. "
                   "직접 조사하거나 평가하지 마라. 한국어로 답하라."),
)

out = team_agent.invoke({"messages": [{"role": "user", "content":
    "CVE-2021-44228 을 조사하고 위험도까지 평가해줘."}]})

print("부른 도구:", tool_names(out))
print()
for m in out["messages"]:
    for c in getattr(m, "tool_calls", None) or []:
        if c["name"] == "task":
            print(f"  {c['args']['subagent_type']:12} <- {str(c['args']['description'])[:62]}")

부른 도구: ['task', 'task']

  researcher   <- CVE-2021-44228을 조사해 한국어로 다음 형식으로 정리해줘: 1) 개요 2) 영향 3) 공격 방식 4)
  analyst      <- CVE-2021-44228에 대해 위험도를 한국어로 평가해줘. 출력 형식은 1) 위험도 등급(높음/보통/낮음) 


`task` 도구가 두 번 불렸고, **누구에게 맡겼는지**(`subagent_type`)가 기록에 남습니다.

### 서브에이전트의 진짜 이점: 대화를 더럽히지 않는다

researcher가 CVE를 조사하며 주고받은 긴 내용은 **그 안에서 끝납니다.**
부모에게는 **결과만** 돌아옵니다.

한 에이전트가 전부 했다면 그 모든 중간 과정이 대화에 쌓여,
세 번째 CVE를 다룰 때쯤에는 컨텍스트가 가득 차고 비용도 불어납니다.

> 12장의 컨텍스트 엔지니어링이 구조로 해결된 사례입니다.

In [10]:
print(answer_text(out)[:400])

CVE-2021-44228은 **위험도 높음**으로 평가됩니다.

### 판단 근거
- **원격 코드 실행(RCE)**로 이어질 수 있는 **치명적 취약점**입니다.
- 환경 정보가 없는 상태에서는, 실제 노출 여부를 배제할 수 없으므로 **보수적으로 최고 수준의 위험**으로 보는 것이 타당합니다.
- Log4j는 다양한 제품과 서비스에 내장되어 있어, **직접 사용 여부를 모르는 경우에도 영향 가능성**이 큽니다.

### 실무 권고사항
- **즉시 Log4j 버전 점검** 및 **패치/업그레이드**를 진행하세요.
- 업그레이드가 어렵다면 임시로:
  - 취약 기능 비활성화
  - 외부 JNDI/LDAP 통신 차단
  - 관련 로그 및 트래픽 모니터링
- 애플리케이션뿐 아니라 **서드파티 제품 내 포함 여


## Step 5. 스킬 — 회사 양식을 파일로 가르치기

"우리 회사 보고서는 표지·요약·상세·조치 계획 순서로, 한 장에 글머리 5개 이하로…"

이런 규칙을 지시문에 다 적으면 **지시문이 끝없이 길어집니다.** 양식이 열 종류면 열 배입니다.
게다가 보고서를 안 쓰는 작업에서도 그 내용이 매번 모델에게 전송됩니다.

**스킬**은 이 규칙을 `SKILL.md` 파일로 빼내는 방법입니다.

In [11]:
SKILL_DIR = WORK / "skills" / "pptx-report"
SKILL_DIR.mkdir(parents=True, exist_ok=True)

(SKILL_DIR / "SKILL.md").write_text("""---
name: pptx-report
description: 보안 월간 보고서를 PowerPoint(.pptx)로 만들 때 사용한다. 슬라이드 구성과 작성 규칙이 들어 있다.
---

# 보안 월간 보고서 PPT 작성 규칙

## 슬라이드 구성 (이 순서를 지킬 것)
1. 표지 — 제목은 "보안 취약점 월간 보고서", 본문 첫 줄에 작성 월
2. 요약 — 전체 CVE 건수와 최고 위험도
3. CVE별 상세 — CVE 하나당 슬라이드 하나. 제목은 CVE 번호
4. 조치 계획 — 위험도 높은 것부터

## 작성 규칙
- 한 슬라이드에 글머리 5개 이하
- 숫자와 버전은 입력 자료에서만 가져온다. 추정 금지
- 저장 경로는 output/report.pptx

## 만드는 방법
make_pptx 도구를 쓴다. slides 는 [{"title": "...", "bullets": ["...", "..."]}, ...] 형태다.
""", encoding="utf-8")

print((SKILL_DIR / "SKILL.md").read_text(encoding="utf-8")[:240])

---
name: pptx-report
description: 보안 월간 보고서를 PowerPoint(.pptx)로 만들 때 사용한다. 슬라이드 구성과 작성 규칙이 들어 있다.
---

# 보안 월간 보고서 PPT 작성 규칙

## 슬라이드 구성 (이 순서를 지킬 것)
1. 표지 — 제목은 "보안 취약점 월간 보고서", 본문 첫 줄에 작성 월
2. 요약 — 전체 CVE 건수와 최고 위험도
3. CVE별 상세 — CVE 하나당 


### 구조와 점진적 공개

```
layer4_work/skills/
└── pptx-report/
    └── SKILL.md     <- 맨 위 --- 사이가 메타데이터(이름·설명)
```

핵심은 **점진적 공개(progressive disclosure)** 입니다.

| 시점 | 모델에게 전달되는 것 |
|---|---|
| 평소 | 이름 + `description` 한 줄만 |
| 필요하다고 판단했을 때 | 파일 전체를 **직접 읽어 감** |

지시문에 다 적으면 항상 전송되지만, 스킬은 **쓸 때만** 읽힙니다.
스킬이 수십 개여도 평소 비용은 목록 몇 줄입니다.

In [12]:
skill_agent = create_deep_agent(
    model=MODEL,
    backend=backend,
    skills=["/skills/"],                    # <- 이 한 줄
    system_prompt="너는 보고서 작성자다. 한국어로 답하라.",
)

out = skill_agent.invoke({"messages": [{"role": "user", "content":
    "지금 쓸 수 있는 스킬이 뭐야? 이름과 설명만 알려줘."}]})
print(answer_text(out)[:260])

지금 쓸 수 있는 스킬은 1개입니다.

- **pptx-report**: 보안 월간 보고서를 PowerPoint(.pptx)로 만들 때 사용한다. 슬라이드 구성과 작성 규칙이 들어 있다.


에이전트가 **스킬의 존재를 알고 있습니다.** 아직 내용은 읽지 않았습니다 — 이름과 설명뿐입니다.

## Step 6. PPT 만드는 도구

스킬은 **무엇을 어떻게 쓸지**를 알려 줄 뿐, 실제로 파일을 만드는 것은 도구입니다.

> **우리가 만드는 도구도 샌드박스를 지켜야 합니다.** 하네스가 준 도구들은
> `root_dir` 밖으로 못 나가지만, 우리 함수는 아무 데나 쓸 수 있습니다.
> 아래 세 줄이 그 구멍을 막습니다.

In [13]:
from pptx import Presentation
from pptx.util import Pt


@tool
def make_pptx(path: str, slides: list[dict]) -> str:
    """슬라이드 목록으로 PowerPoint 파일을 만든다.

    Args:
        path: 저장 경로 (작업 폴더 기준, 예: output/report.pptx)
        slides: [{"title": 제목, "bullets": [글머리, ...]}, ...]
    """
    target = (WORK / path.lstrip("/")).resolve()
    if not str(target).startswith(str(WORK.resolve())):     # 샌드박스 밖이면 거부
        return "거부: 작업 폴더 밖에는 저장할 수 없습니다."

    prs = Presentation()
    for s in slides:
        slide = prs.slides.add_slide(prs.slide_layouts[1])   # 제목 + 내용
        slide.shapes.title.text = s.get("title", "")
        body = slide.placeholders[1].text_frame
        body.clear()
        for i, b in enumerate(s.get("bullets", [])):
            p = body.paragraphs[0] if i == 0 else body.add_paragraph()
            p.text = str(b)
            p.font.size = Pt(18)

    target.parent.mkdir(parents=True, exist_ok=True)
    prs.save(target)
    return f"저장 완료: {path} (슬라이드 {len(slides)}장)"


print("도구 준비:", make_pptx.name)

도구 준비: make_pptx


## Step 7. 전부 합친 하네스

네 가지를 한 번에 켭니다. **인자 네 개가 각각 하나의 개념**입니다.

```python
create_deep_agent(
    backend=...,      # 파일시스템(샌드박스)
    middleware=[...], # planning
    subagents=[...],  # 서브에이전트
    skills=[...],     # 스킬
)
```

In [14]:
harness = create_deep_agent(
    model=MODEL,
    tools=[make_pptx],
    backend=FilesystemBackend(root_dir=str(WORK)),   # 샌드박스
    middleware=[TodoListMiddleware()],                # planning
    subagents=[researcher, analyst],                  # 분업
    skills=["/skills/"],                              # 회사 양식
    system_prompt=("너는 보안팀 리더다. 조사는 researcher, 위험도 평가는 analyst 에게 맡겨라. "
                   "보고서를 만들 때는 반드시 해당 스킬을 먼저 읽고 그 규칙을 따르라. "
                   "한국어로 답하라."),
)

print("harness 준비 완료")

harness 준비 완료


In [15]:
out = harness.invoke({"messages": [{"role": "user", "content":
    "input/cve_list.md 의 CVE 3건을 조사하고 위험도를 평가한 뒤, "
    "pptx-report 스킬에 따라 2026년 9월 보안 월간 보고서를 만들어줘."}]},
    {"recursion_limit": 60})

print("부른 도구:", tool_names(out))

부른 도구: ['read_file', 'write_todos', 'read_file', 'task', 'task', 'task', 'task', 'task', 'task', 'write_todos', 'make_pptx', 'write_todos']


### 할 일 목록은 어떻게 흘러갔나

In [16]:
for t in out.get("todos", []):
    mark = "[x]" if t["status"] == "completed" else "[ ]"
    print(f"  {mark} {t['content']}")

  [x] input/cve_list.md 읽기 및 CVE 3건 확인
  [x] researcher로 각 CVE 조사
  [x] analyst로 각 CVE 위험도 평가
  [x] pptx-report 규칙에 맞춰 슬라이드 구성
  [x] output/report.pptx 생성


### 누구에게 맡겼나

In [17]:
for m in out["messages"]:
    for c in getattr(m, "tool_calls", None) or []:
        if c["name"] == "task":
            print(f"  {c['args']['subagent_type']:12} <- {str(c['args']['description'])[:60]}")

  researcher   <- 조사 대상 CVE-2021-44228에 대해 개요, 영향, 조치사항을 한국어로 간단명료하게 정리해라. 최신 
  researcher   <- 조사 대상 CVE-2024-20337에 대해 개요, 영향, 조치사항을 한국어로 간단명료하게 정리해라. 최신 
  researcher   <- 조사 대상 CVE-2023-4863에 대해 개요, 영향, 조치사항을 한국어로 간단명료하게 정리해라. 최신 공
  analyst      <- 다음 3개 CVE의 위험도를 한국어로 평가하라: CVE-2021-44228, CVE-2024-20337, C
  analyst      <- 다음 3개 CVE의 위험도를 한국어로 평가하라: CVE-2021-44228, CVE-2024-20337, C
  analyst      <- 다음 3개 CVE의 위험도를 한국어로 평가하라: CVE-2021-44228, CVE-2024-20337, C


### 결과물 확인

In [18]:
print("만들어진 파일:")
for p in sorted(WORK.rglob("*")):
    if p.is_file():
        print(f"  {p.relative_to(WORK).as_posix():34} {p.stat().st_size:>8,} bytes")

만들어진 파일:
  input/cve_list.md                       223 bytes
  output/report.pptx                   33,855 bytes
  output/risk.md                        3,169 bytes
  skills/pptx-report/SKILL.md             845 bytes


In [19]:
# PPT 안을 직접 열어 확인한다
prs = Presentation(WORK / "output" / "report.pptx")
print(f"슬라이드 {len(prs.slides)}장\n")
for i, slide in enumerate(prs.slides, 1):
    title = slide.shapes.title.text if slide.shapes.title else "(제목 없음)"
    print(f"{i}. {title}")
    for ph in slide.placeholders:
        if ph.placeholder_format.idx != 0:
            for line in ph.text_frame.paragraphs:
                if line.text:
                    print(f"     - {line.text[:66]}")

슬라이드 6장

1. 보안 취약점 월간 보고서
     - 작성 월: 2026년 9월
     - 점검 대상: CVE 3건
     - 주요 위험도: 높음
     - 대상: Apache Log4j, Cisco AnyConnect, libwebp
     - ※ 사내 사용 버전 기준으로 점검
2. 요약
     - 전체 CVE 건수: 3건
     - 최고 위험도: 높음
     - 높음: 2건
     - 보통: 1건
     - 우선 조치 대상: CVE-2021-44228, CVE-2023-4863
3. CVE-2021-44228
     - Apache Log4j의 JNDI lookup 관련 원격 코드 실행 취약점
     - 악성 문자열이 로그에 기록되면 원격 코드 실행 가능
     - 인터넷 노출 서비스와 사용자 입력 경로가 특히 위험
     - 위험도: 높음
     - 조치: Log4j 업그레이드 및 JNDI lookup 비활성화
4. CVE-2024-20337
     - Cisco AnyConnect / Cisco Secure Client 관련 취약점
     - 사내 사용 버전: 4.9.0
     - 제품 구성과 설치 버전에 따라 영향이 달라질 수 있음
     - 위험도: 보통
     - 조치: Cisco 공식 권고로 영향 여부와 수정 버전 확인
5. CVE-2023-4863
     - libwebp의 WebP 디코딩 힙 기반 버퍼 오버플로우
     - Chrome 및 Chromium 기반 브라우저 등 WebP 처리 제품에 영향 가능
     - 악성 WebP 처리 시 메모리 손상 및 코드 실행 가능
     - 위험도: 높음
     - 조치: libwebp 및 관련 제품의 보안 업데이트 적용
6. 조치 계획
     - 1순위: CVE-2021-44228 대응
     - 2순위: CVE-2023-4863 대응
     - 3순위: CVE-2024-20337 영향 확인 및 업데이트
     - 공통: 번들/정적 포함 라이브러리

**스킬에 적은 네 단계 구성이 그대로 지켜졌습니다** — 표지 → 요약 → CVE별 상세 → 조치 계획.

지시문에는 "스킬을 읽고 따르라"는 한 줄뿐이었습니다. 구성 규칙은 전부 `SKILL.md` 안에 있었고,
에이전트가 필요한 순간에 읽어 갔습니다. **양식을 바꾸려면 그 파일만 고치면 됩니다.**

## Step 8. 정리

### 네 줄로 요약하면

```python
create_deep_agent(
    backend=FilesystemBackend(root_dir="작업폴더"),  # 할 수 있는 범위를 좁힌다
    middleware=[TodoListMiddleware()],              # 길을 잃지 않게 한다
    subagents=[researcher, analyst],                # 일을 나눈다
    skills=["/skills/"],                            # 양식을 파일로 뺀다
)
```

### 언제 무엇이 필요한가

| 상황 | 쓸 것 | 필요 없을 때 |
|---|---|---|
| 단계가 5개 이상 | planning | 두세 단계면 과함 |
| 중간 산출물을 주고받음 | 파일시스템 | 답 한 줄이면 불필요 |
| 성격이 다른 일이 섞임 | 서브에이전트 | 한 가지 일이면 오히려 느려짐 |
| 양식·규칙이 길고 가끔 쓰임 | 스킬 | 항상 쓰는 규칙은 지시문에 |

**전부 켜는 것이 좋은 게 아닙니다.** 각각 비용이 있습니다 —
서브에이전트는 호출이 늘고, planning은 토큰을 더 쓰며, 스킬은 파일 관리가 생깁니다.

### 이 노트북에서 기억할 것

| | |
|---|---|
| 샌드박스는 가장 값싼 안전장치 | `root_dir` 한 줄로 피해 범위가 정해집니다 |
| 우리 도구도 경계를 지켜야 | 하네스가 준 도구만 안전합니다 |
| `description`은 부모용 설명서 | 서브에이전트의 지시문은 부모가 못 봅니다 |
| 서브에이전트는 대화를 지켜 준다 | 중간 과정이 부모에게 쌓이지 않습니다 |
| 스킬은 쓸 때만 읽힌다 | 지시문은 항상 전송됩니다 |
| 양식은 파일로 | 바꿀 때 코드를 건드리지 않습니다 |

### 더 해볼 것

1. `SKILL.md`의 슬라이드 구성을 바꾸고 다시 실행해 보세요. **코드는 한 줄도 바꾸지 않습니다.**
2. `skills/email-summary/SKILL.md`를 하나 더 만들어 보세요. 에이전트가 둘 중 맞는 것을 고르는지 확인해 보세요.
3. `subagents`를 빼고 실행해 전체 메시지 수를 비교해 보세요. 서브에이전트가 대화를 얼마나 아껴 주는지 보입니다.
4. `make_pptx`의 샌드박스 검사 세 줄을 지우고 `../../hack.pptx` 로 저장을 시켜 보세요. 왜 필요한지 체감됩니다.